Tic Tac Toe
---
Two players against each other

<img style="float:left" src="board.png" alt="drawing" width="200"/>

In [14]:
import numpy as np
import pickle
from itertools import product

### Board State
---
Reflect & Judge the state

2 players p1 and p2; p1 uses symbol 1 and p2 uses symbol 2, vacancy as 0

In [67]:
class State:
    def __init__(self,p1,p2):
        self.board = np.zeros((3,3))
        self.p1 = p1
        self.p2 = p2
        self.isEnd = False
        self.boardHash = None
        self.current_player = 1 #1 is p1, -1 is p2

    def available_positions(self):
        pos = []
        for i in range(3):
            for j in range(3):
                if self.board[i,j] == 0:
                    pos.append((i,j))
        return pos
    
    def make_move(self, position):
        if position not in self.available_positions():
            return None
        self.board[position] = self.current_player
        self.current_player = self.current_player*-1

    def get_hash(self):
        self.boardHash = str(self.board.reshape(3 * 3))
        return self.boardHash

    def check_winner(self):
        #check if rows contains 3 or -3 (someone win)
        for i in range(3): 
            if sum(self.board[i,:]) == 3:
                self.isEnd = True
                return 1 #player 1 won
        for i in range(3): #loop on the rows
            if sum(self.board[i,:]) == -3:
                self.isEnd = True
                return -1 #player 2 won
        
        #check if col contains 3 or -3
        for i in range(3):
            if sum(self.board[:,i]) == 3:
                self.isEnd = True
                return 1
        for i in range(3):
            if sum(self.board[:,i]) == -3:
                self.isEnd = True
                return -1
        
        #check diagonal win
        diag_sum = sum([self.board[i,i] for i in range(3)])
        if diag_sum == 3:
            self.isEnd= True
            return 1
        if diag_sum == -3:
            self.isEnd = True
            return -1
        
        diag_sum = sum([self.board[i,3-i-1] for i in range(3)])
        if diag_sum == 3:
            self.isEnd= True
            return 1
        if diag_sum == -3:
            self.isEnd = True
            return -1
        
        #here no one won..
        if len(self.available_positions())==0 :
            self.isEnd = True
            return 0 #no one won
        
        return None #Here there are still moves, so keep playing !!!
    
    def reward(self, result):
        if result == 1:
            self.p1.give_rew(1) #player 1 won, so give 1 reward
            self.p2.give_rew(0)
        elif result == -1:
            self.p1.give_rew(0)
            self.p2.give_rew(1)
        else:
            self.p1.give_rew(0.1) #give a less reward because we don't want ties
            self.p2.give_rew(0.5)

    def reset(self):
        self.board = np.zeros((3, 3))
        self.boardHash = None
        self.isEnd = False
        self.current_player = 1

    def show_board(self):
        # p1: x  p2: o
        for i in range(0, 3):
            print('-------------')
            out = '| '
            for j in range(0, 3):
                if self.board[i, j] == 1:
                    token = 'x'
                if self.board[i, j] == -1:
                    token = 'o'
                if self.board[i, j] == 0:
                    token = ' '
                out += token + ' | '
            print(out)
        print('-------------')    

    def train(self, rounds=100):
        for i in range(rounds):
            if i % 1000 == 0:
                print("Rounds {}".format(i))
            while not self.isEnd:
                
                # Player 1
                positions = self.available_positions()
                p1_action = self.p1.choose_action(positions, self.board, self.current_player)
                # take action and update board state
                self.make_move(p1_action)
                board_hash = self.get_hash()
                self.p1.add_state(board_hash, p1_action)
                # check the board status if it is ended
                win = self.check_winner()
                
                if win is not None: #It returns None only when no one finished or tied.
                    # self.showBoard()
                    # ended with p1 either win or draw
                    self.reward(win) #send rewards to the players, the game has ended
                    self.p1.reset()
                    self.p2.reset()
                    self.reset()
                    break

                else:
                    # Player 2
                    positions = self.available_positions()
                    p2_action = self.p2.choose_action(positions, self.board, self.current_player)
                    self.make_move(p2_action)
                    board_hash = self.get_hash()
                    self.p2.add_state(board_hash, p1_action)

                    win = self.check_winner()
                    if win is not None:
                        # self.showBoard()
                        # ended with p2 either win or draw
                        self.reward(win)
                        self.p1.reset()
                        self.p2.reset()
                        self.reset()
                        break

    def test(self):
        while not self.isEnd:
            # Player 1
            positions = self.available_positions()
            p1_action = self.p1.choose_action(positions, self.board, self.current_player)
            # take action and update board state
            self.make_move(p1_action)
            #self.showBoard()
            # check board status if it is ended
            win = self.check_winner()
            if win is not None: #if win is not None means someone win or tie
                return win

            else:
                # Player 2
                positions = self.available_positions()
                p2_action = self.p2.choose_action(positions, self.board, self.current_player)

                self.make_move(p2_action)
                #self.showBoard()
                win = self.check_winner()
                if win is not None:
                    return win

## Reinforcement Learning Player

In [161]:
class RLPlayer:
    def __init__(self, name, exp_rate = 0.3):
        self.name = name
        self.states = []  # record all positions taken
        self.lr = 0.2
        self.exp_rate = exp_rate
        self.decay_gamma = 0.9
        self.states_value = {}  # state -> value

    def get_hash(self, board):
        boardHash = str(board.reshape(3*3))
        return boardHash

    def add_state(self, state, action):
        self.states.append(state)

    def choose_action(self, positions, current_board, symbol):
        """Return a random action (P = 0.3) or the action with max value (P = 0.7)"""
        if np.random.uniform(0, 1) <= self.exp_rate: # Do exploration, take random 
            # take random action
            idx = np.random.choice(len(positions))
            action = positions[idx]
        else: #Here do exploitation, take the action that has the highest value
            value_max = -999
            for p in positions:
                next_board = current_board.copy() #create a tmp board
                next_board[p] = symbol #do the action
                next_board_hash = self.get_hash(next_board) #get the hash
                value = 0 if self.states_value.get(next_board_hash) is None else self.states_value.get(next_board_hash)
                # print("value", value)
                if value >= value_max: #find the action that has max value. 
                    value_max = value
                    action = p
        return action
    
    def reset(self):
        self.states = []

    def give_rew(self, reward):
        #At the end of the game, I'll get a reward. The iterating on the states in reverse.
        # Set the value of the state to 0 if not existing, otherwise update it with the reward. 
        for st in reversed(self.states):
            if self.states_value.get(st) is None: #if the state doesn't have a value, set it to 0
                self.states_value[st] = 0
            #this is V(t) = V(t) + lr * (gamma*V(t+1) - V(t))
            self.states_value[st] += self.lr * (self.decay_gamma * reward - self.states_value[st])
            reward = self.states_value[st]
            
    def save_policy(self):
        fw = open('policy_' + str(self.name), 'wb')
        pickle.dump(self.states_value, fw)
        fw.close()

    def load_policy(self, file):
        fr = open(file,'rb')
        self.states_value = pickle.load(fr)
        fr.close()

## Q-Learning Player

In [128]:
def crea_dizionario_stati_validi():
    
    stati_validi = generate_valid_tic_tac_toe_states()
    dizionario_stati = {}
    for stato in stati_validi:
        tupla_chiave = tuple(tuple(riga) for riga in stato)
        dizionario_stati[tupla_chiave] = {}
        for i in range(3):
            for j in range(3):
                dizionario_stati[tupla_chiave][(i,j)] = 0

    return dizionario_stati

def generate_valid_tic_tac_toe_states():
    simboli = [0, 1, -1]
    dimensione_griglia = 3
    possibili_combinazioni = product(simboli, repeat=dimensione_griglia**2)
    stati_validi = []

    for combinazione in possibili_combinazioni:
        stato = [list(combinazione[i:i+dimensione_griglia]) for i in range(0, dimensione_griglia**2, dimensione_griglia)]
        if is_valid_state(stato):
            stati_validi.append(stato)
    return stati_validi

def is_valid_state(state):
    count_x = sum(row.count(1) for row in state)
    count_o = sum(row.count(-1) for row in state)

    # Verifica che il numero di 'X' e 'O' sia coerente
    if abs(count_x - count_o) > 1:
        return False

    # Verifica se la partita è finita
    if check_winner(state, 1) and check_winner(state, -1):
        return False  # Entrambi i giocatori non possono vincere contemporaneamente
    if check_winner(state, 1) and count_x == count_o:
        return False  # 'X' non può vincere se è il turno di 'O'
    if check_winner(state, -1) and count_x > count_o:
        return False  # 'O' non può vincere se è il turno di 'X'

    return True

def check_winner(state, player):
    # Verifica le righe, colonne e diagonali per determinare se il giocatore ha vinto
    for i in range(3):
        if all(cell == player for cell in state[i]) or all(state[j][i] == player for j in range(3)):
            return True
    if all(state[i][i] == player for i in range(3)) or all(state[i][2 - i] == player for i in range(3)):
        return True
    return False

{(0, 0): 0, (0, 1): 0, (0, 2): 0, (1, 0): 0, (1, 1): 0, (1, 2): 0, (2, 0): 0, (2, 1): 0, (2, 2): 0}


In [148]:
class QLPlayer:
    def __init__(self, name, exp_rate = 0.3):
        self.name = name
        self.states = []  # record all positions taken + action
        self.lr = 0.2
        self.exp_rate = exp_rate
        self.decay_gamma = 0.9
        self.Q_values = crea_dizionario_stati_validi()

    def get_hash(self, board):
        boardHash = str(board.reshape(3*3))
        return boardHash

    def add_state(self, state, action):
        elementi_str = state.replace('[', '').replace(']', '').split()
        state = [int(float(elemento)) for elemento in elementi_str]
        self.states.append((state,action))

    def choose_action(self, positions, current_board, symbol):
        """Return a random action (P = 0.3) or the action with max value (P = 0.7)"""
        if np.random.uniform(0, 1) <= self.exp_rate: # Do exploration, take random 
            # take random action
            idx = np.random.choice(len(positions))
            action = positions[idx]
        else: #Here do exploitation, take the action that has highest value
            value_max = -999
            for p in positions:
                current_status = tuple(tuple(riga) for riga in current_board)
                next_value = self.Q_values[current_status][p]
                if next_value > value_max:
                    value_max = next_value
                    action = p
        return action

    def reset(self):
        self.states = []

    def give_rew(self, reward):
        #At the end of the game, I'll get a reward. The iterating on the states in reverse.
        # Set the value of the state to 0 if not existing, otherwise update it with the reward. 
        for st in reversed(self.states):
            # Suddivido l'array in sottogruppi di 3 elementi
            sottogruppi = [st[0][i:i+3] for i in range(0, len(st[0]), 3)]
            current_status = tuple(tuple(riga) for riga in sottogruppi)
            action = st[1]
            current_q_value = self.Q_values[current_status][action]
            reward = current_q_value + self.lr * (self.decay_gamma * reward - current_q_value)
            self.Q_values[current_status][action] = round(reward, 3)
    

## Random Player

In [162]:
class RandomPlayer:
    def __init__(self, name):
        self.name = "random"

    def choose_action(self, positions, board, current_player):
        x = np.random.randint(0,len(positions)-1)
        return positions[x]
    
    def add_state(self, state):
        pass

    def give_rew(self, reward):
        pass
            
    def reset(self):
        pass

### Training Reinforcement LEarning

In [163]:
p1 = RLPlayer("p1")
p2 = RLPlayer("p2")

st = State(p1, p2)
print("training...")
st.train(50000)

training...
Rounds 0
Rounds 1000
Rounds 2000
Rounds 3000
Rounds 4000
Rounds 5000
Rounds 6000
Rounds 7000
Rounds 8000
Rounds 9000
Rounds 10000
Rounds 11000
Rounds 12000
Rounds 13000
Rounds 14000
Rounds 15000
Rounds 16000
Rounds 17000
Rounds 18000
Rounds 19000
Rounds 20000
Rounds 21000
Rounds 22000
Rounds 23000
Rounds 24000
Rounds 25000
Rounds 26000
Rounds 27000
Rounds 28000
Rounds 29000
Rounds 30000
Rounds 31000
Rounds 32000
Rounds 33000
Rounds 34000
Rounds 35000
Rounds 36000
Rounds 37000
Rounds 38000
Rounds 39000
Rounds 40000
Rounds 41000
Rounds 42000
Rounds 43000
Rounds 44000
Rounds 45000
Rounds 46000
Rounds 47000
Rounds 48000
Rounds 49000


In [19]:
p1.save_policy()
p2.save_policy()

In [20]:
p1.load_policy("policy_p1")

## Test Reinforcement Learning

In [172]:
p2 = RandomPlayer("Random")
epochs = 1000
st = State(p1,p2)
win_comp = 0

for epoch in range(epochs):
    win = st.test()
    if win == 1:
        win_comp+=1 
    st.reset()

perc = win_comp / epochs * 100
perc

82.39999999999999

## Train Q-Learning

In [150]:
p1 = QLPlayer("p1")
p2 = QLPlayer("p2")

st = State(p1, p2)
print("training...")
st.train(50000)

training...
Rounds 0
Rounds 1000
Rounds 2000
Rounds 3000
Rounds 4000
Rounds 5000
Rounds 6000
Rounds 7000
Rounds 8000
Rounds 9000
Rounds 10000
Rounds 11000
Rounds 12000
Rounds 13000
Rounds 14000
Rounds 15000
Rounds 16000
Rounds 17000
Rounds 18000
Rounds 19000
Rounds 20000
Rounds 21000
Rounds 22000
Rounds 23000
Rounds 24000
Rounds 25000
Rounds 26000
Rounds 27000
Rounds 28000
Rounds 29000
Rounds 30000
Rounds 31000
Rounds 32000
Rounds 33000
Rounds 34000
Rounds 35000
Rounds 36000
Rounds 37000
Rounds 38000
Rounds 39000
Rounds 40000
Rounds 41000
Rounds 42000
Rounds 43000
Rounds 44000
Rounds 45000
Rounds 46000
Rounds 47000
Rounds 48000
Rounds 49000


## Tes Q-Learning

In [158]:
p2 = RandomPlayer("Random")
epochs = 1000
st = State(p1,p2)
win_comp = 0

for epoch in range(epochs):
    win = st.test()
    if win == 1:
        win_comp+=1 
    st.reset()

perc = win_comp / epochs * 100
perc

72.6